### Part 1 - Set up the Evaluation Mental Model
#### Step 1 - Import TensorFlow and Supporting Tools

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

In [12]:
try:
    import tensorflow_datasets as tfds
except ImportError:
    import sys
    import subprocess
    subprocess.call([sys.executable, '-m', 'pip', 'install', 'tensorflow_datasets'])
    import tensorflow_datasets as tfds

#### Step 2 - Review the Three Data Splits


In [13]:
# Loading from Internet
# train_data, val_data, test_data = tfds.load(
#     name='movie_review', split=['train[:12000]', 'test[12000:16000]', 'test[:5000]'], as_supervised=True,
# )

In [14]:
from datasets import load_from_disk

# Load from local disk (no internet needed)
dataset = load_from_disk("./imdb_stanfordnlp")

train_data = dataset["train"].select(range(12000))
val_data = dataset["train"].select(range(12000,16000))
test_data  = dataset["test"].select(range(5000))

train_data = tf.data.Dataset.from_tensor_slices((
        list(train_data["text"]),
        list(train_data["label"])
    ))
val_data = tf.data.Dataset.from_tensor_slices((
        list(val_data["text"]),
        list(val_data["label"])
    ))
test_data  = tf.data.Dataset.from_tensor_slices((
        list(test_data["text"]),
        list(test_data["label"])
    ))

print(train_data)
print(val_data)
print(test_data)

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>


In [15]:
for review, label in train_data.take(2):
    print("Review: ", review.numpy().decode("utf-8")[:700], "...")
    print("Label: ", label.numpy())
    print("-" * 100)

Review:  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ...
Label:  0
----------------------------------------------------------------------------------------------------
Review:  "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on

In [16]:
max_token = 10000
sequence_length = 180

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_token,
    output_mode="int",
    output_sequence_length=sequence_length
)
train_text_only = train_data.map(lambda text, label: text)
vectorizer.adapt(train_text_only.batch(1024))
vocab = vectorizer.get_vocabulary()
print("Vocabulary size: ", len(vocab))
print("First 20 words: ", vocab[:20])

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
Vocabulary size:  20000
First 20 words:  ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('and'), np.str_('to'), np.str_('of'), np.str_('is'), np.str_('in'), np.str_('this'), np.str_('i'), np.str_('it'), np.str_('that'), np.str_('br'), np.str_('was'), np.str_('movie'), np.str_('for'), np.str_('but'), np.str_('with'), np.str_('as')]


In [17]:
for review, label in train_data.take(1):
    one_review = tf.expand_dims(review, axis=0)
    vectorized_review = vectorizer(one_review)
    print("Review: ", review.numpy().decode("utf-8")[:300], "...")
    print("\nVectorized shape: ", vectorized_review.shape)
    print("First 50 token IDs: ")
    print(vectorized_review.numpy()[0][:50])

Review:  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...

Vectorized shape:  (1, 180)
First 50 token IDs: 
[   10  1133    10   214     1    40    62   340  1033    78     6    33
     2  7840    12  2925    11    56    11    14    88   752     8  8853
    10   103   556    12    28    88    11    14 18613    37   202 11246
    42    11   116   600     5  2654     9   837  1475   108     3   357
     6   106]


In [18]:
batch_size = 64

train_batches = (
    train_data.shuffle(12000, seed=42).batch(batch_size).prefetch(tf.data.AUTOTUNE)
)
val_batches = (
    val_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)
)
test_batches = (
    test_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)
)

for batch_review, batch_label in train_batches.take(1):
    print("Batch review shape: ", batch_review.shape)
    print("Batch label shape: ", batch_label.shape)
    print("First 10 tokens: ", batch_label[:10].numpy())

Batch review shape:  (64,)
Batch label shape:  (64,)
First 10 tokens:  [0 0 0 0 0 0 0 0 0 0]


In [19]:
embedding_dim = 64
baseline_model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=max_token, output_dim=embedding_dim, name="baseline_embedding"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
baseline_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (1, 180)               │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ baseline_embedding (Embedding)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
baseline_history = baseline_model.fit(
    train_batches,
    validation_data=val_batches,
    epochs=4,
    shuffle=False,
)

Epoch 1/4
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.9976 - loss: 0.0555 - val_accuracy: 0.1250 - val_loss: 6.4211
Epoch 2/4
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 0.1250 - val_loss: 7.6852
Epoch 3/4
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 1.0000 - loss: 6.3368e-04 - val_accuracy: 0.1250 - val_loss: 8.4408
Epoch 4/4
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 1.0000 - loss: 3.3327e-04 - val_accuracy: 0.1250 - val_loss: 8.9859
